In [1]:
from Bio import SeqIO
import choppy as cp
import primer3
from collections import defaultdict
import re
from functools import lru_cache

In [2]:
all_sequences = list(SeqIO.parse("data/20240414-forSveta.fa", "fasta"))

# seq_names = ["Maizel_COS-AT1G27430-GYF2", "Maizel_COS-SETH5", 
#              "Pereira_COS-SynDNA-f1", "Pereira_COS-SynDNA-f2",
#              "PV252688r_p6utr", "PQ537341r_p6utr,"
#              "PQ488556r_p6utr", "PX021458r_p6utr",
#              "MZ289137_rep", "OR500095r_naive"]

# sequences = [seq for seq in all_sequences if seq.id in seq_names]

sequences = all_sequences

for seq in sequences:
    seq.seq = seq.seq.upper()
    
# I just haven't decided whether I want a list or a dict
sequences_by_id = {seq.id: seq for seq in sequences}

In [3]:

CONFIG = {
    'kmer_size': 15,
    'max_frag_length': 1000,
    'min_frag_length': 200,
    'min_overlap': 50,
    'max_overlap': 100,
    # The model will end the segment after it reaches this length
    # It is also penalized for not reaching it, though it is possible to end prematurely
    'opt_segment_length': 5000, 
    'min_segment_length': 1000,
    # Space for TT1
    'segment_offset_left': 78,
    'segment_offset_right': 108,
    # Space for TT2
    'seq_offset_left': 106,
    'seq_offset_right': 105,
    # Optimal primer length is 20 (hardcoded below), but this range is generally allowed
    'min_primer_length': 17,
    'max_primer_length': 30,
    # Use to search for mispriming, 
    # misprime Tm is calculated only for matching kmer at 3' end of primer
    'primer_3prime_anchor': 6,
    'max_misprime_tm': 47.0,
    # If overlaps don't differ much, that is the distance between them
    # If there is considerable difference in neighbourhood, this parameter is ignored
    'min_step': 10,
    # Standard primer3 parameters
    'min_gc': 0.3,
    'max_gc': 0.7,
    'min_tm': 57.0,
    'max_tm': 62.0,
    'max_hairpin_tm': 24.0,
    'max_homodimer_tm': 45.0,
    'max_heterodimer_tm': 45.0,
    'max_3_self_tm': 35.0,
    # params to match NEB Phusion HF Tm calculator
    'tm_params': dict(
        mv_conc=222.0, dv_conc=0.0, dntp_conc=0.0, dna_conc=500.0,
        tm_method='santalucia', salt_corrections_method='schildkraut',
    ),
    'poly_x_pattern': re.compile(r'(A{5,}|T{5,}|G{5,}|C{5,})'),
    # Due to technical reasons, just having a regexp for CGclamp is not enough
    'clamp_length': 3,
    'clamp_pattern': re.compile(r'[GC][AT][GC]|[AT][GC][GC]'),
    # This is the pattern for primers' 5' end. Currenntly allows anything.
    'five_prime_clamp_pattern': re.compile(r'^[ATGC]'),
    'seg_boundary_pattern': re.compile(r'GC')
}

In [4]:
bg_trie = cp.load_trie("../data/S_cerevisiae-R64-GCA_000146045_cat_15.marisa")
seq_tries = {}
for seq in sequences:
    trie = cp.create_kmer_trie(seq, CONFIG['kmer_size'], bg=False)
    seq_tries[seq.id] = trie 

bg_regions = {}
cur_seq_regions = {}
for seq in sequences:
    bg_regions[seq.id] = cp.find_non_homologous_regions(seq, bg_trie, [], 
                                                        CONFIG['kmer_size'], 
                                                        threshold=CONFIG['min_overlap'])
    cur_seq_regions[seq.id] = cp.find_non_homologous_regions(seq, seq_tries[seq.id], bg_trie, 
                                                             CONFIG['kmer_size'], 
                                                             threshold=CONFIG['min_overlap'])

Processing sequence: 100%|██████████| 5911/5911 [00:00<00:00, 1058514.68it/s]


Found 18 unique k-mers of size 15 in 1 sequence(s).
Constructing a trie...


Processing sequence: 100%|██████████| 8495/8495 [00:00<00:00, 752097.36it/s]


Found 0 unique k-mers of size 15 in 1 sequence(s).
Constructing a trie...


Processing sequence: 100%|██████████| 21124/21124 [00:00<00:00, 1699801.97it/s]


...

Processing sequence: 100%|██████████| 8137/8137 [00:00<00:00, 2385812.77it/s]


Found 2 unique k-mers of size 15 in 1 sequence(s).
Constructing a trie...


Processing sequence: 100%|██████████| 8096/8096 [00:00<00:00, 3185467.65it/s]


Found 4 unique k-mers of size 15 in 1 sequence(s).
Constructing a trie...


Finding non-homologous regions: 100%|██████████| 8096/8096 [00:00<00:00, 518595.05it/s]


## Functions for primer search

(And for local neighborhood construction)

In [5]:
def reverse_complement(seq):
    return seq[::-1].translate(str.maketrans('ATGC', 'TACG'))

# helper search utilities (assume lists are sorted ascending)
def next_val(a, val, default=None):
    return next((x for x in a if x > val), default)

def prev_val(a, val, default=None):
    return next((x for x in reversed(a) if x < val), default)

def compute_homfree_ranges(seq_str, kmer_size):
    """Return a list of (start,end) homfree ranges for each base in seq_str.

    A position i is annotated with the nearest previous/next k-mer collision
    adjusted to k-mer coordinates, matching the original inline logic.
    """
    kmers = defaultdict(list)
    for i in range(len(seq_str) - kmer_size + 1):
        kmer = seq_str[i:i+kmer_size]
        kmers[kmer].append(i)
        kmers[reverse_complement(kmer)].append(i)

    kmers = {k: v for k, v in kmers.items() if len(v) > 1}

    homfree_ranges = [(0, len(seq_str))] * len(seq_str)

    for i in range(len(seq_str)):
        start, end = homfree_ranges[i]
        if i < len(seq_str) - kmer_size + 1:
            kmer = seq_str[i:i+kmer_size]
            if kmer in kmers:
                n_val = next_val(kmers[kmer], i)
                if n_val is not None:
                    end = min(end, n_val + kmer_size - 1)
        if i >= kmer_size - 1:
            kmer = seq_str[i-kmer_size+1:i+1]
            if kmer in kmers:
                p_val = prev_val(kmers[kmer], i - kmer_size + 1)
                if p_val is not None:
                    start = max(start, p_val + 1)
        homfree_ranges[i] = (start, end)

    return homfree_ranges

def build_anchor_kmers(sequences, anchor_len, reverse_position = True):
    """Builds the 3' k-mer hash map for fast off-target screening."""
    anchor_kmers = defaultdict(list)
    for seq in sequences:
        seq_str = str(seq.seq).upper()
        for pos in range(len(seq_str) - anchor_len + 1):
            kmer_fwd = seq_str[pos:pos + anchor_len]
            kmer_rev = reverse_complement(kmer_fwd)
            
            anchor_kmers[kmer_fwd].append((seq.id, pos + anchor_len - 1, "forward"))
            if reverse_position:
                anchor_kmers[kmer_rev].append((seq.id, len(seq_str) - pos - anchor_len, "reverse"))
            else:
                anchor_kmers[kmer_rev].append((seq.id, pos + anchor_len - 1, "reverse"))
            
    return anchor_kmers

def check_misprime(primer_cand, native_3_prime_pos, side, seq_id, anchor_kmers, sequences_by_id, cfg):
    """Validates the primer against the k-mer map to ensure no high-Tm off-targets."""
    anchor_3 = primer_cand[-cfg['primer_3prime_anchor']:]
    
    for anchor_seq_id, anchor_pos, anchor_side in anchor_kmers.get(anchor_3, []):
        # Allow binding to the intended on-target site
        if anchor_side == side and anchor_seq_id == seq_id and anchor_pos == native_3_prime_pos:
            continue
            
        anchor_seq = str(sequences_by_id[anchor_seq_id].seq).upper()
        
        # Extract the off-target sequence depending on strand orientation
        if anchor_side == "forward":
            pos_misprime = anchor_seq[max(0, anchor_pos - cfg['max_primer_length'] + 1):anchor_pos + 1]
        else:
            pos_misprime = reverse_complement(anchor_seq[anchor_pos:min(len(anchor_seq), anchor_pos + cfg['max_primer_length'])])
            
        if primer3.calc_heterodimer_tm(primer_cand, reverse_complement(pos_misprime)) > cfg['max_misprime_tm']:
            return True
            
    return False

def check_local_misprime(primer_cand, seq_str, native_3_prime_pos, side, seq_id, anchor_kmers, cfg):
    """Checks for potential mispriming within the same potential fragment."""
    anchor_3 = primer_cand[-cfg['primer_3prime_anchor']:]
    
    for anchor_seq_id, anchor_pos, anchor_side in anchor_kmers.get(anchor_3, []):
        if anchor_seq_id != seq_id:
            continue
        if anchor_side != side:
            continue
        if anchor_pos == native_3_prime_pos:
            continue
        if anchor_pos - native_3_prime_pos > cfg['max_frag_length'] or native_3_prime_pos > anchor_pos:
            continue

        pos_misprime = seq_str[max(0, anchor_pos - cfg['max_primer_length'] + 1):anchor_pos + 1]
            
        if primer3.calc_heterodimer_tm(primer_cand, reverse_complement(pos_misprime)) > cfg['max_misprime_tm']:
            return True            
    return False

def find_primers_in_regions(seq_record, regions, side, anchor_kmers, sequences_by_id, cfg):
    """Finds primer candidates for a given sequence, regions, and orientation."""
    primer_candidates = []
    seq_id = seq_record.id
    original_seq = str(seq_record.seq).upper()
    
    if side == "forward":
        search_seq = original_seq
        search_regions = regions
    elif side == "reverse":
        search_seq = reverse_complement(original_seq)
        seq_len = len(original_seq)
        search_regions = [(seq_len - r[1], seq_len - r[0]) for r in regions]
    else:
        raise ValueError("Side must be 'forward' or 'reverse'")

    for region in search_regions:
        clamp_matches = cfg['clamp_pattern'].finditer(search_seq, region[0], region[1])
        
        for m in clamp_matches:
            range_start = max(m.start() - (cfg['max_primer_length'] - cfg['clamp_length']), region[0])
            range_end = m.start() - cfg['min_primer_length'] + cfg['clamp_length'] + 1
            
            # Extend 5' -> 3'
            for pr_start in range(range_end - 1, range_start - 1, -1):
                primer_cand = search_seq[pr_start:m.end()]
                if cfg['poly_x_pattern'].search(primer_cand): break
                if not cfg['five_prime_clamp_pattern'].search(primer_cand): continue
                
                gc_content = (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand)
                if gc_content < cfg['min_gc'] or gc_content > cfg['max_gc']: continue
                
                tm = primer3.calc_tm(primer_cand, **cfg['tm_params'])
                if tm > cfg['max_tm']: break
                if tm < cfg['min_tm']: continue
                if primer3.calc_hairpin_tm(primer_cand) > cfg['max_hairpin_tm']: continue
                if primer3.calc_homodimer_tm(primer_cand) > cfg['max_homodimer_tm']: continue
                if primer3.calc_end_stability(primer_cand, primer_cand).tm > cfg['max_3_self_tm']: continue
                
                if side == "forward":
                    native_3_prime_pos = m.end() - 1
                    pos_tuple = (pr_start, m.end())
                else:
                    native_3_prime_pos = len(original_seq) - m.end()
                    pos_tuple = (len(original_seq) - m.end(), len(original_seq) - pr_start)
                
                if check_local_misprime(primer_cand, search_seq, native_3_prime_pos, side, seq_id, anchor_kmers, cfg):
                    break
                    
                primer_candidates.append({
                    'seq': primer_cand,
                    'gc_content': gc_content,
                    'tm': tm,
                    'pos': pos_tuple,
                    'side': side
                })
                
    return primer_candidates

def find_primer_flanked_overlaps(seq_record, primer_regions, overlap_regions, anchor_kmers, sequences_by_id, cfg):
    """Finds all valid primer pairs that flank overlaps within the specified regions."""
    forward_primers = find_primers_in_regions(seq_record, primer_regions, "forward", anchor_kmers, sequences_by_id, cfg)
    reverse_primers = find_primers_in_regions(seq_record, primer_regions, "reverse", anchor_kmers, sequences_by_id, cfg)
    
    primer_flanked_overlaps = []

    for reg in overlap_regions:
        reg_forward_primers = list(filter(lambda x: x['pos'][0] >= reg[0] and x['pos'][1] <= reg[1], forward_primers))
        reg_reverse_primers = list(filter(lambda x: x['pos'][0] >= reg[0] and x['pos'][1] <= reg[1], reverse_primers))
        if len(reg_forward_primers) > 0 and len(reg_reverse_primers) > 0:
            for fwd in reg_forward_primers:
                for rev in reg_reverse_primers:
                    overlap_start = fwd['pos'][0]
                    overlap_end = rev['pos'][1]
                    if overlap_end - overlap_start >= cfg['min_overlap'] and overlap_end - overlap_start <= cfg['max_overlap']:
                        primer_flanked_overlaps.append({
                            'forward': fwd,
                            'reverse': rev,
                            'pos': (overlap_start, overlap_end)
                        })
    return primer_flanked_overlaps

Running the abovedefined functions to get primer-flanked overlaps. They sit inside homology free regions relative to the background. The sequence-related homology is taken into account later.

In [6]:
anchor_kmers = build_anchor_kmers(sequences, CONFIG['primer_3prime_anchor'])

primer_flanked_overlaps = {}

for seq in sequences:
    print(seq.id)
    primer_flanked_overlaps[seq.id] = find_primer_flanked_overlaps(seq, bg_regions[seq.id], bg_regions[seq.id], anchor_kmers, sequences_by_id, CONFIG)


Maizel_COS-AT1G27430-GYF2
Maizel_COS-SETH5
Pereira_COS-SynDNA-f1
Pereira_COS-SynDNA-f2
PV252688r_p6utr
MK050105r_p6utr
MN450855r_p6utr
PQ488560r_p6utr
LC177792r_p6utr
AB890001r_p6utr
MH184583_rep
MG020022r_naive
OR500095r_p6utr
MN450853r_p6utr
JN998607r_p6utr
PQ537341r_p6utr
MZ289137_rep
MZ542728r_p6utr
PQ541186r_p6utr
ON644869r_p6utr
MG020022r_p6utr
MT840363_rep
OP610066r_p6utr
OR500095r_naive
PX021458r_p6utr
PQ488556r_p6utr


Now, getting local neighborhood ranges and constructing the final fragment overlaps

In [7]:
homfree_ranges = {}
for seq in sequences:
    seq_str = str(seq.seq).upper()
    homfree_ranges[seq.id] = compute_homfree_ranges(seq_str, CONFIG['kmer_size'])

def get_overlap_range(start, end, homfree_ranges, kmer_size):
    ov_end = min([el[1] for el in homfree_ranges[start:end - kmer_size + 1]])
    ov_start = max([el[0] for el in homfree_ranges[start + kmer_size - 1:end]])
    return ov_start, ov_end

for seq in sequences:
    seq_id = seq.id
    for ov in primer_flanked_overlaps[seq_id]:
        ov["range"] = get_overlap_range(ov["pos"][0], ov["pos"][1], homfree_ranges[seq_id], CONFIG['kmer_size'])
    primer_flanked_overlaps[seq_id] = [
        ov
        for ov in primer_flanked_overlaps[seq_id]
        if not (
            ov["range"][0] > ov["pos"][1] - CONFIG["min_frag_length"]
            or ov["range"][1] < ov["pos"][0] + CONFIG["min_frag_length"]
        )
    ]

In [8]:
# to bridge the complex repetitive gap we add two sort of primer-flanked, shorter overlaps
# only one of them is expected to be used, but let algorithm decide, which one
seq_id = 'Pereira_COS-SynDNA-f2'

pos = (16411, 16411 + 43)
seq_str = str(sequences_by_id[seq_id].seq)
print(seq_str[pos[0]:pos[1]])
primer_cand = "GTAGTCACATACCTGAAGAGGCAC"
print("forward one:")
print(primer3.calc_tm(primer_cand, **CONFIG['tm_params']))
print(primer3.calc_hairpin_tm(primer_cand))
print(primer3.calc_homodimer_tm(primer_cand))
print(primer3.calc_end_stability(primer_cand, primer_cand).tm)
# Well, that's a very nasty hairpin, to be honest...
forward_pr = {"seq": primer_cand,
                "gc_content": (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand),
                "tm": primer3.calc_tm(primer_cand, **CONFIG['tm_params']),
                "pos": (pos[0], pos[0] + len(primer_cand)),
                "side": "forward"}

primer_cand = reverse_complement("GGCAGAAAGTTTCACCTGTTCT")
print("reverse one:")
print(primer3.calc_tm(primer_cand, **CONFIG['tm_params']))
print(primer3.calc_hairpin_tm(primer_cand))
print(primer3.calc_homodimer_tm(primer_cand))
print(primer3.calc_end_stability(primer_cand, primer_cand).tm)
reverse_pr = {"seq": primer_cand,
                "gc_content": (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand),
                "tm": primer3.calc_tm(primer_cand, **CONFIG['tm_params']),
                "pos": (pos[1] - len(primer_cand), pos[1]),
                "side": "reverse"}
range_ov = get_overlap_range(pos[0], pos[1], homfree_ranges[seq_id], CONFIG['kmer_size'])
print(f"Overlap range: {range_ov}")
primer_flanked_overlaps[seq_id].append({
    "forward": forward_pr,
    "reverse": reverse_pr,
    "pos": (pos[0], pos[1]),
    "range": range_ov
})

pos = (16928, 16928 + 40)
seq_str = str(sequences_by_id[seq_id].seq)
print(seq_str[pos[0]:pos[1]])
primer_cand = "TTACTCACAACATACAGAGAAGCC"
print("forward two:")
print(primer3.calc_tm(primer_cand, **CONFIG['tm_params']))
print(primer3.calc_hairpin_tm(primer_cand))
print(primer3.calc_homodimer_tm(primer_cand))
print(primer3.calc_end_stability(primer_cand, primer_cand).tm)
# Well, that's a very nasty hairpin, to be honest...
forward_pr = {"seq": primer_cand,
                "gc_content": (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand),
                "tm": primer3.calc_tm(primer_cand, **CONFIG['tm_params']),
                "pos": (pos[0], pos[0] + len(primer_cand)),
                "side": "forward"}

primer_cand = reverse_complement("GAGAAGCCGAGTATTTTCTACCAA")
print("reverse two:")
print(primer3.calc_tm(primer_cand, **CONFIG['tm_params']))
print(primer3.calc_hairpin_tm(primer_cand))
print(primer3.calc_homodimer_tm(primer_cand))
print(primer3.calc_end_stability(primer_cand, primer_cand).tm)
reverse_pr = {"seq": primer_cand,
                "gc_content": (primer_cand.count('G') + primer_cand.count('C')) / len(primer_cand),
                "tm": primer3.calc_tm(primer_cand, **CONFIG['tm_params']),
                "pos": (pos[1] - len(primer_cand), pos[1]),
                "side": "reverse"}
range_ov = get_overlap_range(pos[0], pos[1], homfree_ranges[seq_id], CONFIG['kmer_size'])
print(f"Overlap range: {range_ov}")
primer_flanked_overlaps[seq_id].append({
    "forward": forward_pr,
    "reverse": reverse_pr,
    "pos": (pos[0], pos[1]),
    "range": range_ov
})

GTAGTCACATACCTGAAGAGGCACAGAAAGTTTCACCTGTTCT
forward one:
62.121293968297095
44.32879191090444
-46.22747264652193
-42.49151514903241
reverse one:
60.740541141579286
36.73831186622289
3.2172850196243985
5.577948619868437
Overlap range: (0, 18697)
TTACTCACAACATACAGAGAAGCCGAGTATTTTCTACCAA
forward two:
59.99377007478091
32.593277392564346
-20.710488676262344
-126.87021508389176
reverse two:
60.04004866748613
30.896312429318584
-42.90228704672958
-101.04725393346138
Overlap range: (0, 18082)


## Check for hard-to-bridge gaps

In [9]:
gap_threshold = 1.5 * CONFIG['max_frag_length']
gaps = {}

for seq in sequences:
    seq_len = len(seq.seq)
    gaps[seq.id] = []
    
    if primer_flanked_overlaps[seq.id]:
        overlaps = sorted([
            ov['pos'] 
            for ov in primer_flanked_overlaps[seq.id]
        ])
        
        if overlaps[0][0] > gap_threshold:
            gaps[seq.id].append((0, overlaps[0][0]))
        
        for i in range(len(overlaps) - 1):
            gap_start = overlaps[i][1]
            gap_end = overlaps[i + 1][0]
            gap_size = gap_end - gap_start
            if gap_size > gap_threshold:
                gaps[seq.id].append((gap_start, gap_end))
        
        if overlaps[-1][1] < seq_len - gap_threshold:
            gaps[seq.id].append((overlaps[-1][1], seq_len))
    else:
        gaps[seq.id].append((0, seq_len))

for seq in sequences:
    print(f"\n{seq.id}: {len(gaps[seq.id])} gaps found")
    for i, (gap_start, gap_end) in enumerate(gaps[seq.id]):
        print(f"  Gap {i+1}: {gap_start}-{gap_end} (length: {gap_end - gap_start})")


Maizel_COS-AT1G27430-GYF2: 0 gaps found

Maizel_COS-SETH5: 0 gaps found

Pereira_COS-SynDNA-f1: 0 gaps found

Pereira_COS-SynDNA-f2: 0 gaps found

PV252688r_p6utr: 0 gaps found

MK050105r_p6utr: 0 gaps found

MN450855r_p6utr: 0 gaps found

PQ488560r_p6utr: 0 gaps found

LC177792r_p6utr: 0 gaps found

AB890001r_p6utr: 0 gaps found

MH184583_rep: 0 gaps found

MG020022r_naive: 0 gaps found

OR500095r_p6utr: 0 gaps found

MN450853r_p6utr: 0 gaps found

JN998607r_p6utr: 0 gaps found

PQ537341r_p6utr: 0 gaps found

MZ289137_rep: 0 gaps found

MZ542728r_p6utr: 0 gaps found

PQ541186r_p6utr: 0 gaps found

ON644869r_p6utr: 0 gaps found

MG020022r_p6utr: 0 gaps found

MT840363_rep: 0 gaps found

OP610066r_p6utr: 0 gaps found

OR500095r_naive: 0 gaps found

PX021458r_p6utr: 0 gaps found

PQ488556r_p6utr: 0 gaps found


## Segment overlaps

They are suppused to sit in cur_seq_regrions (so to be homology free relative to the background and the current sequence)

In [11]:
segment_overlaps = {}
# 'Free end version' of overlaps, no boundary constraints
# for seq in sequences:
#     print(seq.id)
#     cur_overlaps = []
#     for region in cur_seq_regions[seq.id]:
#         if region[1] - region[0] < CONFIG['max_overlap'] and region[1] - region[0] >= CONFIG['min_overlap']:
#             cur_overlaps.append(region)
#             continue
#         for start in range(region[0], region[1] - CONFIG['max_overlap'] + 1, CONFIG['min_step']):
#             cur_overlaps.append((start, start + CONFIG['max_overlap']))
#         for end in range(region[1], region[0] + CONFIG['max_overlap'] - 1, -CONFIG['min_step']):
#             cur_overlaps.append((end - CONFIG['max_overlap'], end))
#     segment_overlaps[seq.id] = sorted(cur_overlaps, key=lambda x: x[0])

# And this one is for the boundary motif constraints It ignores the min_step
for seq in sequences:
    print(seq.id)
    seq_str = str(seq.seq).upper()
    cur_overlaps = []
    for region in cur_seq_regions[seq.id]:
        for m in re.finditer(CONFIG['seg_boundary_pattern'], seq_str[region[0]:region[1]]):
            left_boundary = region[0] + m.start()
            furthest_allowed = min(region[1], left_boundary + CONFIG['max_overlap'])
            for m2 in re.finditer(CONFIG['seg_boundary_pattern'], seq_str[left_boundary + CONFIG['min_overlap']:furthest_allowed]):
                right_boundary = left_boundary + CONFIG['min_overlap'] + m2.end()
                cur_overlaps.append((left_boundary, right_boundary))
    segment_overlaps[seq.id] = sorted(cur_overlaps, key=lambda x: x[0])

Maizel_COS-AT1G27430-GYF2
Maizel_COS-SETH5
Pereira_COS-SynDNA-f1
Pereira_COS-SynDNA-f2
PV252688r_p6utr
MK050105r_p6utr
MN450855r_p6utr
PQ488560r_p6utr
LC177792r_p6utr
AB890001r_p6utr
MH184583_rep
MG020022r_naive
OR500095r_p6utr
MN450853r_p6utr
JN998607r_p6utr
PQ537341r_p6utr
MZ289137_rep
MZ542728r_p6utr
PQ541186r_p6utr
ON644869r_p6utr
MG020022r_p6utr
MT840363_rep
OP610066r_p6utr
OR500095r_naive
PX021458r_p6utr
PQ488556r_p6utr


In [14]:
for seq_id in segment_overlaps:
    print(f"\n{seq_id}: {len(segment_overlaps[seq_id])} segment overlaps found")



Maizel_COS-AT1G27430-GYF2: 111 segment overlaps found

Maizel_COS-SETH5: 92 segment overlaps found

Pereira_COS-SynDNA-f1: 512 segment overlaps found

Pereira_COS-SynDNA-f2: 847 segment overlaps found

PV252688r_p6utr: 1499 segment overlaps found

MK050105r_p6utr: 1445 segment overlaps found

MN450855r_p6utr: 1429 segment overlaps found

PQ488560r_p6utr: 1477 segment overlaps found

LC177792r_p6utr: 740 segment overlaps found

AB890001r_p6utr: 1108 segment overlaps found

MH184583_rep: 1152 segment overlaps found

MG020022r_naive: 1035 segment overlaps found

OR500095r_p6utr: 1356 segment overlaps found

MN450853r_p6utr: 1251 segment overlaps found

JN998607r_p6utr: 1236 segment overlaps found

PQ537341r_p6utr: 1527 segment overlaps found

MZ289137_rep: 1189 segment overlaps found

MZ542728r_p6utr: 1360 segment overlaps found

PQ541186r_p6utr: 1297 segment overlaps found

ON644869r_p6utr: 1131 segment overlaps found

MG020022r_p6utr: 1053 segment overlaps found

MT840363_rep: 921 segm

## Shortest path algorithm

In [15]:
# This function defines an extra cost for sub-optimal edges. Generally this should be in range of 0 - 0.5 of the extra edge cost
# Can be larger for the too long primers to avoid them, but sometimes they are required
def get_edge_penalty(left_ov, right_ov, 
                     opt_primer_len, max_primer_len_diff, 
                     opt_overlap_len, max_overlap_len_diff, 
                     max_tm_diff=5.0):
    penalty = 0.0
    if left_ov['type'] == 'fr' and right_ov['type'] == 'fr':
        tm_diff = abs(left_ov['tm_left'] - right_ov['tm_right'])
        penalty += tm_diff / max_tm_diff * 0.1
    
    if left_ov['type'] == 'fr':
        primer_len_diff = abs(left_ov['pr_left_len'] - opt_primer_len)
        penalty += primer_len_diff / max_primer_len_diff * 0.1
    if right_ov['type'] == 'fr':
        primer_len_diff = abs(right_ov['pr_right_len'] - opt_primer_len)
        penalty += primer_len_diff / max_primer_len_diff * 0.1
    
    left_overlap_len = left_ov['pos'][1] - left_ov['pos'][0]
    right_overlap_len = right_ov['pos'][1] - right_ov['pos'][0]

    if left_overlap_len == 0:
        left_overlap_len = opt_overlap_len
    if right_overlap_len == 0:
        right_overlap_len = opt_overlap_len

    penalty += abs(left_overlap_len - opt_overlap_len) / max_overlap_len_diff * 0.1
    penalty += abs(right_overlap_len - opt_overlap_len) / max_overlap_len_diff * 0.1

    return round(penalty * 100)

# Check for heterodimers in primers
@lru_cache(maxsize=None)
def _heterodimer_tm(a, b):
    # Order-independent cache key so (a,b) and (b,a) share an entry.
    if a > b:
        a, b = b, a
    return primer3.calc_heterodimer_tm(a, b)

def is_heterodimer_risk(fwd, rev, cfg):
    """True if fwd/rev form a dimer hot enough to matter.

    An *extendable* primer-dimer needs the 3' end of one primer to anneal
    (antiparallel) to the other, i.e. the reverse-complement of its 3' anchor
    must occur as a substring of the partner. This is the same screen used for
    off-target priming, just with the partner primer in place of the template.
    primer3 is only consulted for pairs that pass this gate.
    """
    k = cfg['primer_3prime_anchor']
    suspect = (reverse_complement(fwd[-k:]) in rev or
                reverse_complement(rev[-k:]) in fwd)
    if not suspect:
        return False
    return _heterodimer_tm(fwd, rev) > cfg['max_heterodimer_tm']


# This function puts together all overlaps and adds the start and end dummy overlaps

def _has_boundary_motif(pos, seq, boundary_motif_pattern):
    start, end = pos
    starts_with_motif = boundary_motif_pattern.match(seq, start) is not None
    ends_with_motif = any(m.end() == end for m in boundary_motif_pattern.finditer(seq, 0, end))
    return starts_with_motif and ends_with_motif

def get_all_overlaps(primer_flanked_overlaps, segment_overlaps, seq_len, seq, boundary_motif_pattern):
    all_overlaps = (
        [
            {
                'pos': ov,
                'type': 'seg',
                'range': (0, seq_len)
            }
            for ov in segment_overlaps
        ] +
        [
            {
                'pos': ov['pos'], 
                'type': 'fr', 
                'pr_left_len': len(ov['forward']['seq']),
                'pr_right_len': len(ov['reverse']['seq']),
                'pr_left_seq': ov['forward']['seq'],
                'pr_right_seq': ov['reverse']['seq'],
                'tm_left': ov['forward']['tm'],
                'tm_right': ov['reverse']['tm'],
                'range': ov['range'],
                'boundary_motif': _has_boundary_motif(ov['pos'], seq, boundary_motif_pattern),
            } 
            for i, ov in enumerate(primer_flanked_overlaps)
        ] +
        [
            {
                'pos': (0, 0),
                'type': 'seg',
                'range': (0, seq_len)
            }, 
            {
                'pos': (seq_len, seq_len),
                'type': 'seg',
                'range': (0, seq_len)
            }
        ]
    )
    return sorted(all_overlaps, key=lambda x: x['pos'][0])

In [ ]:
# The search algorithm itself
# TO DO: Use config!!!
def get_overlap_states(all_overlaps, min_length, max_length, min_seg_length, opt_seg_length, seq_len,
                       segment_offset_left, segment_offset_right, max_overlap, min_overlap,
                       seq_offset_left, seq_offset_right):

    # TO DO: move to config, but calculate somehow based on input pars
    opt_primer_len = 20
    max_primer_len_diff = 3

    overlap_states = [{} for _ in range(len(all_overlaps))]

    overlap_states[0][(0, seq_len)] = (0, -1, None) # state: (start_seg, end_range): (weight, predecessor_ind, predecessor_state)

    for i, ov in enumerate(all_overlaps):

        if i % 1000 == 0:
            print(f"Processing overlap {i}/{len(all_overlaps)}")
        if len(overlap_states[i]) == 0:
            continue
        j = i + 1
        too_far = False
        while j < len(all_overlaps) and not too_far:
            next_ov = all_overlaps[j]
            if next_ov['pos'][0] - max(ov['pos'][0], 0) > max_length:
                too_far = True
                continue

            length = next_ov['pos'][1] - max(ov['pos'][0], 0)
            if min_length <= length <= max_length:
                edge_weight = 100 + get_edge_penalty(ov, next_ov, 
                                                    opt_primer_len, max_primer_len_diff, 
                                                    max_overlap, max_overlap - min_overlap)
                # split in two: check if next_ov starts a new segment or if it continues the current one
                # segment start loop
                if next_ov['type'] == 'seg' or (next_ov['range'] == (0, seq_len) and next_ov['boundary_motif']):
                    for state, (cur_weight, _, _) in overlap_states[i].items():
                        if next_ov['range'][0] > state[0] or next_ov['pos'][1] > state[1]:
                            continue
                        seg_len = next_ov['pos'][1] - state[0]
                        if seg_len < min_seg_length:
                            continue

                        #segments must have extra space for flanking stuff
                        extra_length = segment_offset_right
                        if state[0] == ov['pos'][0]:
                            extra_length += segment_offset_left
                        if ov['pos'] == (0, 0):
                            extra_length += seq_offset_left
                        if next_ov['pos'] == (seq_len, seq_len):
                            extra_length += seq_offset_right
                        if length + extra_length > max_length:
                            continue

                        extra_penalty = max(opt_seg_length - seg_len, 0) / opt_seg_length * 500

                        new_weight = cur_weight + edge_weight + extra_penalty
                        new_state = (next_ov['pos'][0], next_ov['range'][1])

                        existing = overlap_states[j].get(new_state)
                        if existing is None or new_weight < existing[0]:
                            overlap_states[j][new_state] = (new_weight, i, state)

                # continue segment loop
                if next_ov['type'] == 'fr':
                    for state, (cur_weight, _, _) in overlap_states[i].items():
                        if next_ov['range'][0] > state[0] or next_ov['pos'][1] > state[1]:
                            continue
                        if next_ov['pos'][1] - state[0] > opt_seg_length:
                            # in this case only start segment option is possible
                            continue
                        extra_length = 0
                        if ov['pos'] == (0, 0):
                            extra_length += seq_offset_left
                        if next_ov['pos'] == (seq_len, seq_len):
                            extra_length += seq_offset_right
                        if state[0] == ov['pos'][0]:
                            extra_length += segment_offset_left
                        
                        if length + extra_length > max_length:
                            # if we are at the start of the segment, we need extra space on the left
                            continue
                        # This may take forever...
                        
                        if ov.has_key('pr_left_seq') and next_ov.has_key('pr_right_seq') and is_heterodimer_risk(ov['pr_left_seq'], next_ov['pr_right_seq'], CONFIG):
                            continue
                        new_weight = cur_weight + edge_weight
                        new_state = (state[0], min(next_ov['range'][1], state[1]))

                        existing = overlap_states[j].get(new_state)
                        if existing is None or new_weight < existing[0]:
                            overlap_states[j][new_state] = (new_weight, i, state)
                    
            j += 1
    return overlap_states

Run the search

In [19]:
full_overlap_states = {}
full_all_overlaps  = {}
for seq in sequences:
    print(seq.id)
    full_all_overlaps[seq.id] = get_all_overlaps(primer_flanked_overlaps[seq.id], 
                                                 segment_overlaps[seq.id], 
                                                 len(str(seq.seq)),
                                                 str(seq.seq),
                                                 CONFIG['seg_boundary_pattern'])

    full_overlap_states[seq.id] = get_overlap_states(full_all_overlaps[seq.id], CONFIG['min_frag_length'], 
                                                     CONFIG['max_frag_length'], CONFIG['min_segment_length'], 
                                                     CONFIG['opt_segment_length'], len(str(seq.seq)),
                                                     CONFIG['segment_offset_left'], CONFIG['segment_offset_right'], 
                                                     CONFIG['max_overlap'], CONFIG['min_overlap'],
                                                     CONFIG['seq_offset_left'], CONFIG['seq_offset_right'])


Maizel_COS-AT1G27430-GYF2
Processing overlap 0/3459


KeyError: 'pr_left_seq'